In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import matplotlib.pyplot as plt 
import seaborn as sns 

X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv").squeeze()
y_test = pd.read_csv('data/y_test.csv').squeeze()

print("Data Loaded!")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Data Loaded!
X_train shape: (1362, 9)
X_test shape: (341, 9)


In [2]:
from xgboost import XGBClassifier
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)
print("Model trained successfully!")
print("Number of trees built:", model.n_estimators)

Model trained successfully!
Number of trees built: 100


In [3]:
y_pred = model.predict(X_test)
print("=== MODEL PERFORMANCE ===")
print(classification_report(y_test, y_pred, target_names=['Current', 'Resolved', 'To Be Discontinued']))

=== MODEL PERFORMANCE ===
                    precision    recall  f1-score   support

           Current       1.00      1.00      1.00       229
          Resolved       1.00      1.00      1.00         8
To Be Discontinued       1.00      1.00      1.00       104

          accuracy                           1.00       341
         macro avg       1.00      1.00      1.00       341
      weighted avg       1.00      1.00      1.00       341



In [4]:
feature_columns = [
    'generic_name_encoded',
    'shortage_reason_encoded',
    'dosage_form_encoded',
    'company_name_encoded',
    'update_type_encoded',
    'therapeutic_category_encoded',
    'year',
    'month'
]

X_train_fixed = X_train[feature_columns]
X_test_fixed = X_test[feature_columns]

model2 = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)
model2.fit(X_train_fixed, y_train)
y_pred2 = model2.predict(X_test_fixed)

print("=== MODEL PERFORMANCE (fixed) ===")
print(classification_report(y_test, y_pred2,
      target_names=['Current', 'Resolved', 'To Be Discontinued']))

=== MODEL PERFORMANCE (fixed) ===
                    precision    recall  f1-score   support

           Current       1.00      1.00      1.00       229
          Resolved       1.00      1.00      1.00         8
To Be Discontinued       1.00      1.00      1.00       104

          accuracy                           1.00       341
         macro avg       1.00      1.00      1.00       341
      weighted avg       1.00      1.00      1.00       341



In [5]:
# Time based split - train on data before 2024, test on 2024+
df_model = pd.read_csv('data/df_clean.csv')

# Rebuild features
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

text_columns = ['generic_name', 'availability', 'shortage_reason', 
                'dosage_form', 'company_name', 'update_type', 
                'therapeutic_category']

for col in text_columns:
    df_model[col + '_encoded'] = le.fit_transform(df_model[col])

# Time based split
train = df_model[df_model['year'] < 2024]
test = df_model[df_model['year'] >= 2024]

feature_cols = ['generic_name_encoded', 'shortage_reason_encoded',
                'dosage_form_encoded', 'company_name_encoded',
                'therapeutic_category_encoded', 'year', 'month']

X_train_t = train[feature_cols]
X_test_t = test[feature_cols]
y_train_t = le.fit_transform(train['status'])
y_test_t = le.fit_transform(test['status'])

print("Train size:", X_train_t.shape)
print("Test size:", X_test_t.shape)
print("\nTrain years:", sorted(train['year'].unique()))
print("Test years:", sorted(test['year'].unique()))

Train size: (1132, 7)
Test size: (571, 7)

Train years: [np.int64(2012), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Test years: [np.int64(2024), np.int64(2025), np.int64(2026)]


In [6]:
model3 = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)

model3.fit(X_train_t, y_train_t)
y_pred3 = model3.predict(X_test_t)

print("=== REAL MODEL PERFORMANCE ===")
print(classification_report(y_test_t, y_pred3))

=== REAL MODEL PERFORMANCE ===
              precision    recall  f1-score   support

           0       0.05      1.00      0.10        28
           1       0.43      0.33      0.38        27
           2       0.00      0.00      0.00       516

    accuracy                           0.06       571
   macro avg       0.16      0.44      0.16       571
weighted avg       0.02      0.06      0.02       571



C:\Users\neham\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\neham\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\neham\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [7]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# Encode target ONCE using full dataset
le_target = LabelEncoder()
le_target.fit(df_model['status'])

y_train_t = le_target.transform(train['status'])
y_test_t = le_target.transform(test['status'])

print("Class mapping:")
for i, cls in enumerate(le_target.classes_):
    print(f"{i} = {cls}")

print("\nTrain distribution:", pd.Series(y_train_t).value_counts().to_dict())
print("Test distribution:", pd.Series(y_test_t).value_counts().to_dict())

# Retrain
model3 = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)

model3.fit(X_train_t, y_train_t)
y_pred3 = model3.predict(X_test_t)

print("\n=== REAL MODEL PERFORMANCE ===")
print(classification_report(y_test_t, y_pred3,
      target_names=le_target.classes_))

Class mapping:
0 = Current
1 = Resolved
2 = To Be Discontinued

Train distribution: {0: 1115, 1: 14, 2: 3}
Test distribution: {2: 516, 0: 28, 1: 27}

=== REAL MODEL PERFORMANCE ===
                    precision    recall  f1-score   support

           Current       0.05      1.00      0.10        28
          Resolved       0.18      0.11      0.14        27
To Be Discontinued       0.00      0.00      0.00       516

          accuracy                           0.05       571
         macro avg       0.08      0.37      0.08       571
      weighted avg       0.01      0.05      0.01       571



C:\Users\neham\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\neham\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\neham\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [8]:
df_model['risk']= df_model['status'].apply( lambda x: 1 if x == 'To Be Discountinued' else 0)
train = df_model[df_model['year'] < 2024]
test  = df_model[df_model['year'] >= 2024]

print("Train risk distribution:")
print(train['risk'].value_counts())
print("\nTest risk distribution:")
print(test['risk'].value_counts())

Train risk distribution:
risk
0    1132
Name: count, dtype: int64

Test risk distribution:
risk
0    571
Name: count, dtype: int64


In [9]:
print(df_model['status'].value_counts())
print("\nUnique values:")
print(df_model['status'].unique())

status
Current               1143
To Be Discontinued     519
Resolved                41
Name: count, dtype: int64

Unique values:
<StringArray>
['Current', 'To Be Discontinued', 'Resolved']
Length: 3, dtype: str


In [11]:
df_model['status'] = df_model['status'].str.strip()
df_model['risk']= df_model['status'].apply( lambda x: 1 if x == 'To Be Discountinued' else 0)

print("Risk distribution overall:")
print(df_model['risk'].value_counts())

train = df_model[df_model['year'] < 2024]
test  = df_model[df_model['year'] >= 2024]

print("Train risk distribution:")
print(train['risk'].value_counts())
print("\nTest risk distribution:")
print(test['risk'].value_counts())

Risk distribution overall:
risk
0    1703
Name: count, dtype: int64
Train risk distribution:
risk
0    1132
Name: count, dtype: int64

Test risk distribution:
risk
0    571
Name: count, dtype: int64


In [12]:
sample = df_model[df_model['status'] == 'To Be Discontinued'].head(3)
print("Sample To Be Discontinued records:")
print(sample[['status','risk']])

val = df_model['status'].iloc[2]
print("\nStatus value:", repr(val))
print("Length:", len(val))
print("Matches:", val == 'To Be Discontinued')

Sample To Be Discontinued records:
               status  risk
2  To Be Discontinued     0
5  To Be Discontinued     0
7  To Be Discontinued     0

Status value: 'To Be Discontinued'
Length: 18
Matches: True


In [13]:
df_model = df_model.drop(columns=['risk'])
df_model['risk'] = (df_model['status'] == 'To Be Discontinued').astype(int)
print("Risk Distribution:")
print(df_model['risk'].value_counts())

Risk Distribution:
risk
0    1184
1     519
Name: count, dtype: int64


In [14]:
train = df_model[df_model['year'] < 2024]
test = df_model[df_model['year'] >= 2024]
feature_cols= ['generic_name_encoded', 'shortage_reason_encoded',
                'dosage_form_encoded', 'company_name_encoded',
                'therapeutic_category_encoded', 'year', 'month']

X_train_b = train[feature_cols]
X_test_b = test[feature_cols]
y_train_b = train['risk']
y_test_b = test['risk']

print("Train distribution:")
print(y_train_b.value_counts())
print("\nTest distribution:")
print(y_test_b.value_counts())

Train distribution:
risk
0    1129
1       3
Name: count, dtype: int64

Test distribution:
risk
1    516
0     55
Name: count, dtype: int64
